# 03 Neural stream inventory and decoder manifest

This notebook bridges the trial catalog and the first decoding notebook.

## Purpose
1. Open every NWB file listed in the session index.
2. Inventory acquisition and processing objects across sessions.
3. Identify candidate neural streams for decoding.
4. Build a decoder manifest that maps each session to a likely neural source.
5. Save paper-style figures, tables, and metadata for the baseline decoder notebook.

> **Reference:** Issar et al. (2020) *J Neurophysiol* 123:1472–1485.  
> All figures in this notebook follow the aesthetic conventions of the reference paper:
> overlaid waveform traces, scatter-over-time with regression, dual-axis line plots,
> session-distribution histograms, and shaded mean ± SE curves.

## Inputs
- `/kaggle/working/tables_session_index/session_master_index.csv`
- `/kaggle/working/tables_session_index/decoder_trial_table.csv`
- Raw NWB files under: `/kaggle/input/datasets/katakuricharlotte/dandi-dataset/001201/sub-Monkey-N`

## Outputs
- `tables_stream_inventory/all_object_manifest.csv`
- `tables_stream_inventory/candidate_stream_manifest.csv`
- `tables_stream_inventory/session_decoder_stream_map.csv`
- `meta_stream_inventory/stream_inventory_report.json`
- `figures_stream_inventory/*.png / *.pdf`

In [ ]:
!pip install pynwb h5py --quiet

In [ ]:
import os
import re
import json
import math
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

warnings.filterwarnings("ignore")
from pynwb import NWBHDF5IO

In [ ]:
# ─────────────────────────────────────────────
# PAPER-STYLE GLOBAL SETTINGS
# Mirrors Issar et al. 2020 (J Neurophysiol)
# ─────────────────────────────────────────────
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "axes.linewidth": 1.2,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "xtick.minor.size": 3,
    "ytick.minor.size": 3,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "legend.fontsize": 11,
    "legend.frameon": True,
    "legend.edgecolor": "0.4",
    "grid.linestyle": ":",
    "grid.linewidth": 0.7,
    "grid.alpha": 0.85,
})

# Paper palette (from Issar et al. Fig. 4–6)
C_BLACK   = "#1a1a1a"
C_MAROON  = "#8B2222"   # % waveforms removed line
C_BLUE    = "#2166AC"   # threshold crossings reference
C_GREEN   = "#1B7837"   # maximum decoding
C_GRAY    = "#888888"   # chance / control
C_ORANGE  = "#E08214"   # SpikingBandPower stream
C_RED_FILL = "#D6604D"  # NAS-helps shading

MARKER_PE  = "o"        # monkey Pe / primary subject
MARKER_WA  = "s"        # monkey Wa / secondary subject

def paper_axes(ax, top=True, right=True):
    ax.minorticks_on()
    ax.grid(True, which="major", linestyle=":", linewidth=0.8, alpha=0.85)
    ax.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.6)
    for spine in ax.spines.values():
        spine.set_linewidth(1.2)
    ax.tick_params(which="both", direction="in", top=top, right=right)

np.random.seed(42)

## Paths and notebook dependencies

This notebook expects the outputs from `02_session_index_and_trial_catalog.ipynb` to already exist.

In [ ]:
DATASET_DIR = Path("/kaggle/input/datasets/katakuricharlotte/dandi-dataset/001201/sub-Monkey-N")

IN_TABLE_DIR = Path("/kaggle/input/datasets/katakuricharlotte/02-session-index-results/tables_session_index")
SESSION_MASTER_CSV    = IN_TABLE_DIR / "session_master_index.csv"
DECODER_TRIAL_TABLE_CSV = IN_TABLE_DIR / "decoder_trial_table.csv"

OUT_FIG_DIR   = Path("/kaggle/working/figures_stream_inventory")
OUT_TABLE_DIR = Path("/kaggle/working/tables_stream_inventory")
OUT_META_DIR  = Path("/kaggle/working/meta_stream_inventory")

OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
OUT_META_DIR.mkdir(parents=True, exist_ok=True)

# Sample NWB for deep-dive waveform figures
SAMPLE_NWB = DATASET_DIR / "sub-Monkey-N_ses-20200127_ecephys.nwb"

print("DATASET_DIR         :", DATASET_DIR)
print("SESSION_MASTER_CSV  :", SESSION_MASTER_CSV)
print("DECODER_TRIAL_TABLE :", DECODER_TRIAL_TABLE_CSV)
print("SAMPLE_NWB          :", SAMPLE_NWB)